# logit-thermo A2-20：第 1 段の npz を小分けにする（CPU）

- **必要な secret：なし**。**ランタイム：CPU**（GPU はいりません）。
- `A2_20_s1/` の npz（Qwen2.5-7B・Mistral-7B の 2 つ、各 9MB くらい）を、Claude が取りに行ける大きさ（3MB ずつ）に切って `A2_20_s1/parts/` に置きます。中身は見ません・表示しません。元のファイルのハッシュと、切ったものをつなげたハッシュが同じかを確かめます。
- 1 分くらいで終わります。終わったら、またはエラーで止まったら、通知（🚀・🎉・⚠）を送ってランタイムを切ります。


In [ ]:
import os, time, json, hashlib, urllib.request, traceback
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-20 第1段のデータを小分けにする（CPU、Claude が取りに行けるように）'
def notify(msg, tags='pig'):
    try:
        urllib.request.urlopen(urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                               headers={'Title': 'logit-thermo - A2-20 split stage1 npz', 'Tags': tags}), timeout=10).read()
    except Exception as e:
        print('通知は送れませんでした：', type(e).__name__)
notify('🚀 スタートしました', tags='rocket')
from google.colab import drive, runtime
try:
    drive.mount('/content/drive')
    D = '/content/drive/MyDrive/logit-thermo/A2_20_s1'
    os.makedirs(D + '/parts', exist_ok=True)
    man = {}
    for name in ('qwen2.5-7b@s1.npz', 'mistral-7b-v0.3@s1.npz'):
        raw = open(f'{D}/{name}', 'rb').read()
        parts = [raw[i:i + 3_000_000] for i in range(0, len(raw), 3_000_000)]
        for i, p in enumerate(parts):
            open(f'{D}/parts/{name}.part{i}', 'wb').write(p)
        back = b''.join(open(f'{D}/parts/{name}.part{i}', 'rb').read() for i in range(len(parts)))
        assert hashlib.sha256(back).hexdigest() == hashlib.sha256(raw).hexdigest(), name
        man[name] = dict(size=len(raw), sha256=hashlib.sha256(raw).hexdigest(), parts=[len(p) for p in parts])
    json.dump(man, open(D + '/parts/manifest.json', 'w'), indent=1)
    drive.flush_and_unmount()
    notify('🎉 終わりました：' + ', '.join(f'{k} を {len(v["parts"])} 個に' for k, v in man.items()), tags='tada')
except Exception as e:
    notify('⚠️ 止まりました：\n' + ''.join(traceback.format_exception(type(e), e, e.__traceback__))[-3000:], tags='warning')
    raise
finally:
    time.sleep(3); runtime.unassign()
